# TDQEQ — Fine-Tune Model with LLaMA-Factory

This notebook fine-tunes `Qwen/Qwen2.5-3B-Instruct` on your generated table-normalization dataset using LoRA (or 4-bit QLoRA on T4) via LLaMA-Factory, then validates the result with vLLM.

## Cell 1 — Mount Drive & Detect GPU

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (no GPU)'
print(f"GPU detected: {GPU}")
print("Tip: Use 4-bit QLoRA (USE_4BIT_QLORA = True) on T4 (16GB). Disable on A100/L4.")

## Cell 2 — Install LLaMA-Factory & Dependencies

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e ".[torch,metrics]"
!pip install -q wandb vllm==0.7.2

## Cell 3 — Configuration

Fill in your values. All other cells use these variables.

In [ ]:
# ── USER CONFIGURATION ───────────────────────────────────────────────────────
GDRIVE_DATA_DIR  = "/gdrive/MyDrive/tdqeq-finetune/datasets"
GDRIVE_MODEL_DIR = "/gdrive/MyDrive/tdqeq-finetune/models"
BASE_MODEL_ID    = "Qwen/Qwen2.5-3B-Instruct"
LORA_RANK        = 64

# Safe: ~4,300 tokens per training example (SYSTEM_PROMPT ~1500 + HTML ~2250 + JSON output ~550)
CUTOFF_LEN       = 8192

# Set True for T4 (16GB VRAM), False for A100/L4 (40+GB VRAM)
USE_4BIT_QLORA   = True

NUM_EPOCHS       = 3
WANDB_PROJECT    = "tdqeq-table-finetune"
HF_REPO_ID       = ""          # Optional: fill to push adapter to HuggingFace Hub
# ─────────────────────────────────────────────────────────────────────────────

## Cell 4 — Register Dataset with LLaMA-Factory

Writes a `dataset_info.json` to your Google Drive data directory and points the YAML config to it via `dataset_dir`. This avoids patching LLaMA-Factory's internal files (which breaks on re-clone).

In [ ]:
import json, os

os.makedirs(GDRIVE_MODEL_DIR, exist_ok=True)

dataset_info = {
    "tdqeq_train": {
        "file_name": os.path.join(GDRIVE_DATA_DIR, "train.json"),
        "columns": {
            "prompt":   "instruction",
            "query":    "input",
            "response": "output",
            "system":   "system",
            "history":  "history"
        }
    },
    "tdqeq_val": {
        "file_name": os.path.join(GDRIVE_DATA_DIR, "val.json"),
        "columns": {
            "prompt":   "instruction",
            "query":    "input",
            "response": "output",
            "system":   "system",
            "history":  "history"
        }
    }
}

dataset_info_path = os.path.join(GDRIVE_DATA_DIR, "dataset_info.json")
with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)

print(f"Dataset info written to: {dataset_info_path}")

## Cell 5 — Write Training YAML Config

In [ ]:
quantization_line = 'quantization_bit: 4' if USE_4BIT_QLORA else '# quantization disabled (A100 mode)'
push_to_hub_lines = (
    f"push_to_hub: true\nexport_hub_model_id: \"{HF_REPO_ID}\"\nhub_private_repo: true\nhub_strategy: checkpoint"
    if HF_REPO_ID else
    '# push_to_hub: false'
)

config = f"""
### model
model_name_or_path: {BASE_MODEL_ID}
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: {LORA_RANK}
lora_target: all
{quantization_line}

### dataset
dataset_dir: {GDRIVE_DATA_DIR}
dataset: tdqeq_train
eval_dataset: tdqeq_val
template: qwen
cutoff_len: {CUTOFF_LEN}
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: {GDRIVE_MODEL_DIR}
logging_steps: 10
save_steps: 200
plot_loss: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: {NUM_EPOCHS}
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: tdqeq-table-finetune

{push_to_hub_lines}
"""

config_path = '/content/LLaMA-Factory/examples/train_lora/tdqeq_finetune.yaml'
with open(config_path, 'w') as f:
    f.write(config.strip())

print(f"YAML config written to: {config_path}")
print(config)

## Cell 6 — Run Fine-Tuning

In [ ]:
!cd LLaMA-Factory && llamafactory-cli train examples/train_lora/tdqeq_finetune.yaml

## Cell 7 — Validate Fine-Tuned Model with vLLM

> **This step is required, not optional.** It validates the adapter produces valid JSON before you integrate it into `tdqeq`. Do not skip it.

This cell starts a vLLM server with the LoRA adapter loaded, then sends a test table and verifies the output is well-formed JSON matching the expected schema.

In [ ]:
import subprocess, time

# Start vLLM server in background with the LoRA adapter
server_cmd = [
    "vllm", "serve", BASE_MODEL_ID,
    "--dtype=half",
    "--gpu-memory-utilization", "0.85",
    "--max-lora-rank", str(LORA_RANK),
    "--enable-lora",
    "--lora-modules", f"tdqeq-table={GDRIVE_MODEL_DIR}",
]
proc = subprocess.Popen(server_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("vLLM server starting... waiting 45 seconds.")
time.sleep(45)

In [ ]:
import requests, json, json_repair

# Send a real test table
test_html = ("<table><tr><th>Part No</th><th>Capacitance</th><th>Voltage</th></tr><tr><td>C1005X5R1E104K050BC</td><td rowspan='2'>100nF</td><td>25V</td></tr><tr><td>C1005X5R1C104K050BC</td><td>16V</td></tr></table>")
test_input = [{"html": test_html, "page_number": 1, "confidence_score": 0.97}]

FINETUNE_SYSTEM_MSG = (
    "You are a deterministic data transformation engine. "
    "Convert the input HTML table JSON array into the specified nested JSON structure. "
    "Output only valid JSON."
)

resp = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "tdqeq-table",
        "messages": [
            {"role": "system", "content": FINETUNE_SYSTEM_MSG},
            {"role": "user",   "content": json.dumps(test_input)},
        ],
        "temperature": 0.0,
    },
    timeout=60,
)

raw_output = resp.json()["choices"][0]["message"]["content"]
parsed = json_repair.loads(raw_output)

print("=== RAW MODEL OUTPUT ===")
print(raw_output)
print("\n=== PARSED JSON ===")
print(json.dumps(parsed, indent=2, ensure_ascii=False))

# Validate expected keys are present
if isinstance(parsed, list) and parsed:
    required_keys = {"column_mapping", "data"}
    first = parsed[0]
    missing = required_keys - set(first.keys())
    if missing:
        print(f"\nWARNING: Missing expected keys in output: {missing}")
    else:
        print("\nVALIDATION PASSED: Output contains all required keys.")
        print(f"Integration: set TDQEQ_OPENAI_BASE_URL=http://localhost:8000/v1 and TDQEQ_OPENAI_MODEL=tdqeq-table")
else:
    print("\nWARNING: Output is not a valid non-empty list. Check model training.")